# Notebook 02 — Target Generation

## Purpose
Generates the three model-specific target sequences that RDO optimizes against:

| Target | Prompt type | Intervention | Meaning |
|--------|-------------|-------------|--------|
| `t_answer` | Harmful `p_harm` | DIM **ablation** (all layers) | What the model says when its refusal is removed |
| `t_refusal` | Harmless `p_safe` | DIM **addition** (at `best_layer`) | What refusal looks like on benign input |
| `t_retain` | Harmless `p_safe` | None (clean) | Normal helpful output to preserve |

**Why generate with DIM rather than fixed strings?**  
The paper uses model-specific targets because refusal phrasing differs across models. DIM is an effective existing attack, so it already knows what the model would say under ablation — making it a reliable bootstrap.

**Key infrastructure:** This notebook uses `nnsight` for interventions, not HuggingFace hooks.

**Prerequisite:** Notebook 01 must have been run (data splits saved), and your existing DIM notebook must have saved `dim_direction.pt` and `dim_direction_metadata.json`.

**Outputs saved to `data/saladbench_splits/targets/`:**
- `harmful_targets.json` — contains `{prompt, ablation}` for each training prompt
- `harmless_targets.json` — contains `{prompt, addition, retain}` for each training prompt

In [ ]:
import json
import os

import matplotlib.pyplot as plt
import numpy as np
import torch
from nnsight import LanguageModel
from tqdm.auto import tqdm

os.makedirs('data/saladbench_splits/targets', exist_ok=True)

# ── Configuration ────────────────────────────────────────────────────────────
MODEL_PATH        = 'meta-llama/Llama-3.1-8B-Instruct'  # Primary target model
DIM_DIR_PATH      = 'dim_outputs'                        # Where your existing notebook saved DIM artifacts
NUM_TARGET_TOKENS = 30                                   # Paper: 30 tokens for t_answer and t_refusal
RETAIN_TOKENS     = 29                                   # Paper: 29 tokens for t_retain
BATCH_SIZE        = 4                                    # Reduce if OOM; paper uses 512 but that needs A100

print(f'Model: {MODEL_PATH}')
print(f'Target tokens: {NUM_TARGET_TOKENS} (answer/refusal), {RETAIN_TOKENS} (retain)')

## 1. Load Model via nnsight
`nnsight.LanguageModel` wraps a HuggingFace model and makes its internal activations accessible and differentiable inside `model.trace()` contexts. This is required for the RDO training loop (Notebook 03) but we also use it here for clean, consistent interventions.

In [ ]:
model = LanguageModel(
    MODEL_PATH,
    device_map='auto',
    torch_dtype=torch.float16,
)
model.requires_grad_(False)   # Freeze all model weights — only r will be trained

# Sanity check: run a simple trace
with model.trace('Hello, world!') as tracer:
    pass

print(f'Model loaded: {MODEL_PATH}')
print(f'Hidden size  : {model.config.hidden_size}')
print(f'Num layers   : {model.config.num_hidden_layers}')
print(f'Vocab size   : {model.config.vocab_size}')

## 2. Load DIM Direction
Load the DIM direction and metadata that your existing notebook computed.  
If you saved them with different names, update the paths below.

In [ ]:
direction_file = os.path.join(DIM_DIR_PATH, 'direction.pt')
metadata_file  = os.path.join(DIM_DIR_PATH, 'direction_metadata.json')

if not os.path.exists(direction_file):
    raise FileNotFoundError(
        f'DIM direction not found at {direction_file}.\n'
        'Save your refusal_dir tensor from the existing notebook:\n'
        '    torch.save(refusal_dir, "dim_outputs/direction.pt")\n'
        '    json.dump({"layer": 14, "pos": -1}, open("dim_outputs/direction_metadata.json", "w"))'
    )

best_refusal_direction = torch.load(direction_file).to(model.dtype)
metadata               = json.load(open(metadata_file))
best_layer             = metadata['layer']
best_token_pos         = metadata['pos']
alpha                  = best_refusal_direction.norm().detach().clone()

print(f'DIM direction shape : {best_refusal_direction.shape}')
print(f'DIM direction norm  : {alpha:.4f}  (this is alpha for activation addition)')
print(f'Best layer          : {best_layer}')
print(f'Best token position : {best_token_pos}')

# The unit-normalized version used for ablation
dim_hat = best_refusal_direction / best_refusal_direction.norm()

In [ ]:
# Visualize the DIM direction magnitude across layers (if mean_diffs.pt is available)
mean_diffs_file = os.path.join(DIM_DIR_PATH, 'mean_diffs.pt')
if os.path.exists(mean_diffs_file):
    mean_diffs = torch.load(mean_diffs_file)  # shape: (num_layers, d_model)
    norms = mean_diffs.float().norm(dim=-1).cpu().numpy()
    
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(range(len(norms)), norms, 'o-', markersize=4, linewidth=1.5, color='steelblue')
    ax.axvline(best_layer, color='tomato', linestyle='--', linewidth=1.5,
               label=f'Selected layer {best_layer} (DIM peak)')
    ax.set_xlabel('Layer index')
    ax.set_ylabel('||mean_diff|| (DIM direction norm)')
    ax.set_title('DIM direction norm across layers\n(higher = stronger harmful/harmless separation at that layer)')
    ax.legend()
    plt.tight_layout()
    plt.savefig('data/saladbench_splits/targets/dim_norms_by_layer.png', dpi=150)
    plt.show()
else:
    print('mean_diffs.pt not found — skipping layer-norm visualization.')

## 3. Chat Template
The paper uses model-specific chat templates. The template wraps each instruction before feeding it to the model.

In [ ]:
LLAMA3_CHAT_TEMPLATE = (
    '<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n'
    '{instruction}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n'
)

def apply_chat_template(instructions: list[str]) -> list[str]:
    model_path_lower = MODEL_PATH.lower()
    if 'llama-3' in model_path_lower:
        return [LLAMA3_CHAT_TEMPLATE.format(instruction=inst) for inst in instructions]
    raise ValueError(f'Add a chat template for {MODEL_PATH}')

# Preview
example = apply_chat_template(['Write a poem about the ocean.'])[0]
print('Chat template preview:')
print(repr(example))

In [ ]:
# Load training data from Notebook 01
harmful_train  = json.load(open('data/saladbench_splits/harmful_train.json'))
harmless_train = json.load(open('data/saladbench_splits/harmless_train.json'))

harmful_train_instructions  = apply_chat_template([d['instruction'] for d in harmful_train])
harmless_train_instructions = apply_chat_template([d['instruction'] for d in harmless_train])

print(f'Harmful train  : {len(harmful_train_instructions):,}')
print(f'Harmless train : {len(harmless_train_instructions):,}')

## 4. Helper: Generation Functions

Three generation modes, each using nnsight:
- **`generate_with_ablation`** — subtracts the DIM direction projection from activations at every layer and every token. Used to get `t_answer`.
- **`generate_with_addition`** — adds `alpha * dim_hat` to layer `best_layer` input. Used to get `t_refusal`.
- **`generate_clean`** — no intervention. Used to get `t_retain`.

In [ ]:
def projection(activation, direction):
    """Project activation onto direction: (act · dir_hat) * dir_hat."""
    direction_hat = direction / direction.norm()
    # activation: (batch, seq, d_model), direction_hat: (d_model,)
    proj_scalar = torch.einsum('bsd,d->bs', activation, direction_hat)   # (batch, seq)
    return torch.einsum('bs,d->bsd', proj_scalar, direction_hat)          # (batch, seq, d_model)


def generate_with_ablation(model, prompts, direction, max_new_tokens=30, batch_size=4):
    """Generate completions with DIM direction ablated at ALL layers."""
    completions = []
    direction_hat = (direction / direction.norm()).to(model.dtype)
    
    for i in tqdm(range(0, len(prompts), batch_size), desc='Ablation generation'):
        batch = prompts[i:i+batch_size]
        # nnsight: generate with intervention
        with model.generate(batch, max_new_tokens=max_new_tokens, do_sample=False) as generator:
            for layer in model.model.layers:
                # Ablate from layer input (residual stream entering each layer)
                layer.input[:] = layer.input - projection(layer.input, direction_hat)
            out = generator.output.save()
        
        for j, prompt in enumerate(batch):
            prompt_len = len(model.tokenizer(prompt, add_special_tokens=True)['input_ids'])
            new_tokens = out[j][prompt_len:]
            completions.append(model.tokenizer.decode(new_tokens, skip_special_tokens=True))
    
    return completions


def generate_with_addition(model, prompts, direction, alpha, add_layer_idx,
                           max_new_tokens=30, batch_size=4):
    """Generate completions with DIM direction added at a single layer."""
    completions = []
    direction_hat = (direction / direction.norm()).to(model.dtype)
    
    for i in tqdm(range(0, len(prompts), batch_size), desc='Addition generation'):
        batch = prompts[i:i+batch_size]
        with model.generate(batch, max_new_tokens=max_new_tokens, do_sample=False) as generator:
            model.model.layers[add_layer_idx].input[:] = (
                model.model.layers[add_layer_idx].input + alpha * direction_hat
            )
            out = generator.output.save()
        
        for j, prompt in enumerate(batch):
            prompt_len = len(model.tokenizer(prompt, add_special_tokens=True)['input_ids'])
            new_tokens = out[j][prompt_len:]
            text = model.tokenizer.decode(new_tokens, skip_special_tokens=True)
            # Paper: truncate refusal target at first sentence boundary
            text = text.split('.')[0] if '.' in text else text
            completions.append(text)
    
    return completions


def generate_clean(model, prompts, max_new_tokens=29, batch_size=4):
    """Generate completions without any intervention."""
    completions = []
    
    for i in tqdm(range(0, len(prompts), batch_size), desc='Clean generation'):
        batch = prompts[i:i+batch_size]
        with model.generate(batch, max_new_tokens=max_new_tokens, do_sample=False) as generator:
            out = generator.output.save()
        
        for j, prompt in enumerate(batch):
            prompt_len = len(model.tokenizer(prompt, add_special_tokens=True)['input_ids'])
            new_tokens = out[j][prompt_len:]
            completions.append(model.tokenizer.decode(new_tokens, skip_special_tokens=True))
    
    return completions

print('Generation functions defined.')

## 5. Quick Sanity Check on 3 Examples
Before generating the full dataset (which takes ~hours), verify all three intervention types work correctly on a few examples.

In [ ]:
N_CHECK = 3
check_harmful  = harmful_train_instructions[:N_CHECK]
check_harmless = harmless_train_instructions[:N_CHECK]

print('=== Generating sanity check examples (3 of each type) ===\n')

check_ablation  = generate_with_ablation(model, check_harmful,  best_refusal_direction, max_new_tokens=30, batch_size=1)
check_addition  = generate_with_addition(model, check_harmless, best_refusal_direction, alpha, best_layer, max_new_tokens=30, batch_size=1)
check_retain    = generate_clean(model, check_harmless, max_new_tokens=29, batch_size=1)

for i in range(N_CHECK):
    print(f'--- Example {i} ---')
    print(f'[HARMFUL PROMPT]    {harmful_train[i]["instruction"][:80]}...')
    print(f'[t_answer]          {check_ablation[i][:120]}')
    print()
    print(f'[HARMLESS PROMPT]   {harmless_train[i]["instruction"][:80]}')
    print(f'[t_refusal]         {check_addition[i][:120]}')
    print(f'[t_retain]          {check_retain[i][:120]}')
    print()

**What to look for in the output above:**
- `t_answer` should look like a cooperative response to the harmful prompt (no refusal)
- `t_refusal` should look like a refusal even though the prompt is harmless (e.g., "I cannot...")
- `t_retain` should look like a normal, helpful answer to the harmless prompt

If any of these look wrong, check that `best_layer` and `alpha` are correct.

## 6. Generate Full Target Sets
Results are cached to disk — subsequent runs will load from cache automatically.

In [ ]:
harmful_targets_path  = 'data/saladbench_splits/targets/harmful_targets.json'
harmless_targets_path = 'data/saladbench_splits/targets/harmless_targets.json'

# ── Harmful targets (t_answer) ────────────────────────────────────────────────
if os.path.exists(harmful_targets_path):
    print(f'Loading cached harmful targets from {harmful_targets_path}')
    harmful_targets = json.load(open(harmful_targets_path))
else:
    print('Generating harmful targets (DIM ablation on harmful prompts)...')
    ablation_completions = generate_with_ablation(
        model, harmful_train_instructions,
        best_refusal_direction,
        max_new_tokens=NUM_TARGET_TOKENS,
        batch_size=BATCH_SIZE
    )
    harmful_targets = [
        {'prompt': p, 'ablation': a}
        for p, a in zip(harmful_train_instructions, ablation_completions)
    ]
    with open(harmful_targets_path, 'w') as f:
        json.dump(harmful_targets, f, indent=2)
    print(f'Saved {len(harmful_targets):,} harmful targets.')

print(f'Harmful targets loaded: {len(harmful_targets):,}')

In [ ]:
# ── Harmless targets (t_refusal + t_retain) ───────────────────────────────────
if os.path.exists(harmless_targets_path):
    print(f'Loading cached harmless targets from {harmless_targets_path}')
    harmless_targets = json.load(open(harmless_targets_path))
else:
    print('Generating harmless targets...')
    print('  Step 1/2: Activation addition (t_refusal)...')
    addition_completions = generate_with_addition(
        model, harmless_train_instructions,
        best_refusal_direction, alpha, best_layer,
        max_new_tokens=NUM_TARGET_TOKENS,
        batch_size=BATCH_SIZE
    )
    print('  Step 2/2: Clean generation (t_retain)...')
    retain_completions = generate_clean(
        model, harmless_train_instructions,
        max_new_tokens=RETAIN_TOKENS,   # 29 tokens, not 30
        batch_size=BATCH_SIZE
    )
    harmless_targets = [
        {'prompt': p, 'addition': add, 'retain': ret}
        for p, add, ret in zip(harmless_train_instructions, addition_completions, retain_completions)
    ]
    with open(harmless_targets_path, 'w') as f:
        json.dump(harmless_targets, f, indent=2)
    print(f'Saved {len(harmless_targets):,} harmless targets.')

print(f'Harmless targets loaded: {len(harmless_targets):,}')

## 7. Visualize Target Quality

### 7a. First-token distribution: does DIM ablation actually unlock harmful behavior?

In [ ]:
def get_first_token(text):
    """Return normalized first word (lowercased, stripped)."""
    return text.strip().lower().split()[0] if text.strip() else '<empty>'

# First token of t_answer (should NOT start with refusal words like 'i', 'sorry')
answer_first_tokens  = [get_first_token(t['ablation']) for t in harmful_targets]
# First token of t_refusal (should start with refusal words)
refusal_first_tokens = [get_first_token(t['addition']) for t in harmless_targets]

REFUSAL_WORDS = {'i', 'sorry', 'i\'m', "i'm", 'unfortunately', 'i cannot', 'i\'m unable', 'as'}

answer_refusal_rate  = sum(1 for t in answer_first_tokens  if t in REFUSAL_WORDS) / len(answer_first_tokens)
induced_refusal_rate = sum(1 for t in refusal_first_tokens if t in REFUSAL_WORDS) / len(refusal_first_tokens)

print(f't_answer  — starts with refusal word: {answer_refusal_rate:.1%}  (lower is better)')
print(f't_refusal — starts with refusal word: {induced_refusal_rate:.1%}  (higher is better)')

# Plot first-token frequency for t_answer
from collections import Counter
top_answer  = Counter(answer_first_tokens).most_common(12)
top_refusal = Counter(refusal_first_tokens).most_common(12)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, top, title, color in zip(
        axes,
        [top_answer, top_refusal],
        ['t_answer: first token (from DIM ablation on harmful prompts)',
         't_refusal: first token (from DIM addition on harmless prompts)'],
        ['tomato', 'steelblue']):
    words, cnts = zip(*top)
    ax.bar(words, cnts, color=color, edgecolor='white')
    ax.set_title(title, fontsize=10)
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('data/saladbench_splits/targets/first_token_distribution.png', dpi=150)
plt.show()

In [ ]:
# Show 5 complete target triplets for qualitative inspection
print('=== QUALITATIVE TARGET INSPECTION (5 examples) ===\n')
indices = list(range(min(5, len(harmful_targets))))
for idx in indices:
    h_rec = harmful_targets[idx]
    s_rec = harmless_targets[idx]
    print(f'Example {idx}')
    print(f'  [p_harm]    {harmful_train[idx]["instruction"][:80]}...')
    print(f'  [t_answer]  {h_rec["ablation"][:120]}')
    print(f'  [p_safe]    {harmless_train[idx]["instruction"][:80]}')
    print(f'  [t_refusal] {s_rec["addition"][:120]}')
    print(f'  [t_retain]  {s_rec["retain"][:120]}')
    print()

## 8. Summary

- `harmful_targets.json`: `{prompt, ablation}` — `ablation` is `t_answer`  
- `harmless_targets.json`: `{prompt, addition, retain}` — `addition` is `t_refusal`, `retain` is `t_retain`

**Next step:** Run **Notebook 03** (`03_rdo_training.ipynb`) which uses these targets to run the gradient-based RDO optimization loop.